# Route Optimisation: Last-Mile Delivery on Hybrid Hardware

This notebook solves **Travelling Salesman** (TSP) and **Vehicle Routing** (VRP)
problems via QUBO/Ising formulations on the uniqx platform:

| Workload | Key op | Hardware |
|:---------|:-------|:---------|
| TSP ground state | `eigs` (Ising Hamiltonian) | GPU at scale |
| VRP annealing | `expv` + `expect` (quantum annealing) | GPU / QPU |
| Iterative annealing | `expv` loop (uniqx iterations) | GPU / QPU |

We compare 3 city layouts (Manhattan grid, London ring, warehouse cluster)
and show how problem size drives the CPU → GPU → QPU hardware decision.

**All computation goes through uniqx.**

In [ ]:
import os
from uniqx import connect

# Default to a local service for development.
# For prod, export UNIQX_GATEWAY=api.oriqx.com:443 and UNIQX_API_KEY=...
endpoint = os.environ.get("UNIQX_GATEWAY", "localhost:50050")
client = connect(endpoint)
print("Connected to", endpoint)
import time
import matplotlib.pyplot as plt

from uniqx.domains.optimization.logistics import (
    DeliveryProblem,
    CITY_PRESETS,
    build_routing_module,
    build_vrp_module,
    build_annealing_module,
    decode_route,
)
from uniqx import parse_result
from uniqx.core.execution import connect, preflight, submit, get

API_KEY = os.environ.get("UNIQX_API_KEY")


## 1. City Layouts

In [ ]:
problems = {}
for name, preset in CITY_PRESETS.items():
    prob = DeliveryProblem.from_preset(name)
    problems[name] = prob
    D = prob.distance_matrix()
    total_dist = sum(
        D[i][j] for i in range(prob.n_stops) for j in range(i + 1, prob.n_stops)
    )
    print(f"{name:>20}: {prob.n_stops} stops, total pairwise dist = {total_dist:.1f}")
    print(f"{'':>20}  {preset['description']}")

# Also create random cities
for n in [6, 8]:
    prob = DeliveryProblem.random_city(n_stops=n, seed=42)
    problems[f"random_{n}"] = prob
    print(f"{'random_' + str(n):>20}: {prob.n_stops} stops")

## 2. TSP Ground State: Preflight Comparison

In [ ]:
def print_opts(label, options):
    print(f"\n--- {label} ---")
    if not options:
        return
    for o in options:
        f = " *" if o.get("recommended") else ""
        print(
            f"  {o['label']:<20} time={o['total_time']:>8.1f}  cost=${o['total_cost_usd']:.4f}  err={o['max_error_rate']:.4f}{f}"
        )


tsp_results = {}

for name in ["warehouse_cluster", "london_ring", "random_6"]:
    prob = problems[name]
    mod, inputs, meta = build_routing_module(prob)
    opts = preflight(mod, client=client)
    print_opts(f"TSP: {name} ({prob.n_stops} stops, dim={meta['dim']})", opts)

    tsp_results[name] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["eigenvalues", "eigenvectors"])
        evals = out.get("eigenvalues", ([], "", []))[2] if out else []
        evecs = out.get("eigenvectors", ([], "", []))[2] if out else []

        route = decode_route(evecs, prob.n_stops) if evecs else []
        ground_E = evals[0] if evals else 0.0

        tsp_results[name][opt["label"]] = {
            "time": wall,
            "eigenvalues": evals,
            "route": route,
            "ground_energy": ground_E,
            "option": opt,
        }
        print(f"  {opt['label']}: {wall:.2f}s, E_0={ground_E:.4f}, route={route}")

## 3. Vehicle Routing Problem (Quantum Annealing)

In [ ]:
vrp_results = {}

for name in ["warehouse_cluster", "london_ring"]:
    prob = problems[name]
    mod, inputs, meta = build_vrp_module(prob, n_vehicles=2)
    opts = preflight(mod, client=client)
    print_opts(f"VRP: {name} (2 vehicles, dim={meta['dim']})", opts)

    vrp_results[name] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["final", "energy"])
        energy = out.get("energy", ([], "", [0.0]))[2][0] if out else 0.0

        vrp_results[name][opt["label"]] = {
            "time": wall,
            "energy": energy,
            "option": opt,
        }
        print(f"  {opt['label']}: {wall:.2f}s, energy={energy:.4f}")

## 4. Iterative Annealing (uniqx Loop)

In [ ]:
anneal_results = {}

for n_steps in [10, 20, 40]:
    prob = problems["warehouse_cluster"]
    mod, inputs, meta = build_annealing_module(prob, n_anneal_steps=n_steps)
    opts = preflight(mod, client=client)
    print_opts(f"Annealing: {n_steps} steps (dim={meta['dim']})", opts)

    anneal_results[n_steps] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0

        anneal_results[n_steps][opt["label"]] = {"time": wall, "option": opt}
        print(f"  {opt['label']}: {wall:.2f}s")

## 5. Problem Size Scaling

In [ ]:
scaling = []
for n_stops in [4, 6, 8]:
    prob = DeliveryProblem.random_city(n_stops=n_stops, seed=42)
    mod, inputs, meta = build_routing_module(prob)
    opts = preflight(mod, client=client)
    print(f"\nn_stops={n_stops} (Ising dim={meta['dim']}):")
    for opt in opts:
        f = " *" if opt.get("recommended") else ""
        print(
            f"  {opt['label']:<20} time={opt['total_time']:>8.1f}  cost=${opt['total_cost_usd']:.4f}{f}"
        )
        scaling.append(
            {
                "n_stops": n_stops,
                "dim": meta["dim"],
                "label": opt["label"],
                "time": opt["total_time"],
                "cost": opt["total_cost_usd"],
                "recommended": opt.get("recommended", False),
            }
        )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_hw = {"cpu-only": "#2563eb", "cpu+gpu": "#16a34a", "cpu+gpu+qpu": "#9333ea"}

# Top-left: City layouts with routes
for i, (name, prob) in enumerate(
    [
        ("warehouse_cluster", problems["warehouse_cluster"]),
        ("london_ring", problems["london_ring"]),
    ]
):
    locs = prob.locations
    xs = [l[0] for l in locs]
    ys = [l[1] for l in locs]
    marker = "o" if i == 0 else "s"
    axes[0, 0].scatter(xs, ys, s=80, marker=marker, label=name, zorder=5)
    axes[0, 0].scatter([xs[0]], [ys[0]], s=150, marker="*", color="red", zorder=6)
    # Draw route if available
    for hw_label, data in tsp_results.get(name, {}).items():
        route = data.get("route", [])
        if len(route) >= 2:
            for k in range(len(route)):
                a, b = route[k], route[(k + 1) % len(route)]
                if a < len(locs) and b < len(locs):
                    axes[0, 0].plot(
                        [locs[a][0], locs[b][0]],
                        [locs[a][1], locs[b][1]],
                        "--",
                        alpha=0.4,
                        color="gray",
                    )
            break
axes[0, 0].set_xlabel("x")
axes[0, 0].set_ylabel("y")
axes[0, 0].set_title("City Layouts")
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.3)

# Top-right: TSP execution time by layout
layout_names = list(tsp_results.keys())
for hw in sorted(set(l for n in tsp_results for l in tsp_results[n])):
    times = [tsp_results[n].get(hw, {}).get("time", 0) for n in layout_names]
    c = colors_hw.get(hw, "#6b7280")
    axes[0, 1].bar(
        [
            i
            + list(sorted(set(l for n in tsp_results for l in tsp_results[n]))).index(
                hw
            )
            * 0.25
            for i in range(len(layout_names))
        ],
        times,
        width=0.25,
        color=c,
        label=hw,
        alpha=0.8,
    )
axes[0, 1].set_xticks(range(len(layout_names)))
axes[0, 1].set_xticklabels(layout_names, fontsize=7, rotation=15)
axes[0, 1].set_ylabel("Wall time (s)")
axes[0, 1].set_title("TSP: Execution by Layout")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(axis="y", alpha=0.3)

# Bottom-left: Annealing step scaling
steps_list = sorted(anneal_results.keys())
for hw in sorted(set(l for s in anneal_results for l in anneal_results[s])):
    times = [anneal_results[s].get(hw, {}).get("time", 0) for s in steps_list]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 0].plot(steps_list, times, "o-", color=c, label=hw)
axes[1, 0].set_xlabel("Annealing steps")
axes[1, 0].set_ylabel("Wall time (s)")
axes[1, 0].set_title("Iterative Annealing: Step Scaling")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# Bottom-right: Problem size scaling
hw_labels = sorted(set(d["label"] for d in scaling))
for hw in hw_labels:
    sub = [d for d in scaling if d["label"] == hw]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 1].semilogy(
        [d["n_stops"] for d in sub], [d["time"] for d in sub], "o-", color=c, label=hw
    )
axes[1, 1].set_xlabel("Number of stops")
axes[1, 1].set_ylabel("Estimated time (log)")
axes[1, 1].set_title("TSP: Hardware Scaling with Problem Size")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(alpha=0.3)

fig.suptitle("Route Optimisation on Hybrid Hardware", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

| Problem | Encoding | Ising dim | Best hardware |
|:--------|:---------|:----------|:--------------|
| TSP (6 stops) | QUBO → Ising | ~256 | CPU/GPU |
| TSP (8 stops) | QUBO → Ising (truncated) | 256 | GPU |
| VRP (2 vehicles) | QUBO + capacity | same as TSP | GPU |
| Iterative annealing | expv loop (20 steps) | ~256 | GPU / QPU |

The QUBO → Ising encoding maps combinatorial optimisation problems onto
quantum Hamiltonians. uniqx's cost model routes small instances to
CPU and larger ones to GPU/QPU. Iterative annealing via uniqx's
loop mode enables multi-step quantum annealing in a single submission.